In [ ]:
import torch
import time

# ✅ 检测设备
if not torch.cuda.is_available():
    print("未检测到 GPU，将使用 CPU 运行")
    device = torch.device("cpu")
else:
    device = torch.device("cuda")
    print(f"使用 GPU: {torch.cuda.get_device_name(0)}")

# ⚙️ 调整此参数控制负载大小（越大负载越高）
SIZE = 4096

a = torch.randn(SIZE, SIZE, device=device)
b = torch.randn(SIZE, SIZE, device=device)

print("开始 GPU 压力循环...")
print("点击 Jupyter 工具栏的 ■ Stop 按钮，或按 I I（连按两次 I）可中断\n")

iteration = 0
start_time = time.time()

try:
    while True:
        c = torch.matmul(a, b)
        c = torch.relu(c)
        c = torch.matmul(c, a)
        c = torch.sigmoid(c)

        if device.type == "cuda":
            torch.cuda.synchronize()

        iteration += 1
        elapsed = time.time() - start_time

        if iteration % 10 == 0:
            print(f"迭代: {iteration:6d} | 耗时: {elapsed:.1f}s | 速率: {iteration/elapsed:.2f} iter/s")

except KeyboardInterrupt:
    print(f"\n✅ 已停止。共完成 {iteration} 次迭代，总耗时 {time.time()-start_time:.1f} 秒。")

使用 GPU: NVIDIA A100-SXM4-80GB
开始 GPU 压力循环...
点击 Jupyter 工具栏的 ■ Stop 按钮，或按 I I（连按两次 I）可中断

迭代:     10 | 耗时: 0.2s | 速率: 61.97 iter/s
迭代:     20 | 耗时: 0.3s | 速率: 62.00 iter/s
迭代:     30 | 耗时: 0.5s | 速率: 62.01 iter/s
迭代:     40 | 耗时: 0.6s | 速率: 62.02 iter/s
迭代:     50 | 耗时: 0.8s | 速率: 62.02 iter/s
迭代:     60 | 耗时: 1.0s | 速率: 62.02 iter/s
迭代:     70 | 耗时: 1.1s | 速率: 62.02 iter/s
迭代:     80 | 耗时: 1.3s | 速率: 62.03 iter/s
迭代:     90 | 耗时: 1.5s | 速率: 62.03 iter/s
迭代:    100 | 耗时: 1.6s | 速率: 62.03 iter/s
迭代:    110 | 耗时: 1.8s | 速率: 62.03 iter/s
迭代:    120 | 耗时: 1.9s | 速率: 62.03 iter/s
迭代:    130 | 耗时: 2.1s | 速率: 62.03 iter/s
迭代:    140 | 耗时: 2.3s | 速率: 62.03 iter/s
迭代:    150 | 耗时: 2.4s | 速率: 62.03 iter/s
迭代:    160 | 耗时: 2.6s | 速率: 62.03 iter/s
迭代:    170 | 耗时: 2.7s | 速率: 62.03 iter/s
迭代:    180 | 耗时: 2.9s | 速率: 62.03 iter/s
迭代:    190 | 耗时: 3.1s | 速率: 62.03 iter/s
迭代:    200 | 耗时: 3.2s | 速率: 62.03 iter/s
迭代:    210 | 耗时: 3.4s | 速率: 62.03 iter/s
迭代:    220 | 耗时: 3.5s | 速率: 62.03 iter/s
迭代:    2

# 1-OCR Extraction

In [2]:
import os
import pytesseract
from PIL import Image

In [3]:
pytesseract.pytesseract.tesseract_cmd = "/home/lu.dong1/.conda/envs/expo-judge/bin/tesseract"
print(pytesseract.get_tesseract_version())

5.5.2


In [4]:
image_path = "data/temp/X00016469612.jpg"

In [5]:
output_path = "data/temp/option2"
os.makedirs(output_path, exist_ok=True)

In [6]:
base_name = os.path.splitext(os.path.basename(image_path))[0]
raw_text_path = os.path.join(output_path, f"{base_name}_raw_text.txt")

In [6]:
image = Image.open(image_path)
raw_text = pytesseract.image_to_string(image)

In [9]:
print(raw_text)

tan woon yann

BOOK TA -K (TAMAN DAYA) SDN BHD
TBO7-W
NO.5? $5,57 & 59, JALAN SAGU 18,
TAMAN DAYA
81100 JOHOR BAHRU.
JOHOR.

LAM MCAT

Document Ho : TDO1167104

Date 25/12/2018 8:13:39 PM
Cashier MANIS
Member
CASH BILL
CODE/DESC PRICE Dise AMOUNT
Quy RM RM
9556939040118 AF MODELLING CLAY KIDDY FiSHt
1PC + 9.00) 0,00 9.00
Total : 9,00
Rour ding Adjustment 0.00
Round. :d Total (RM): 9.00
Cash a 40.00.
CHANGE 00

GOODS SOLD ARE NOT RETURNAP
EXCHANGEABLE

THANK YOU
PLEASE COME AGAIN t



In [10]:
with open(raw_text_path, "w", encoding="utf-8") as f:
    f.write(raw_text)

# 2-Model Extraction

In [7]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
import json
import os
import re
import time

/home/lu.dong1/.conda/envs/expo-judge/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
engineA = "/projects/insightx-lab/cn_grpo/models/Llama-3.1-8B-Instruct"

In [9]:
engineB = "/projects/insightx-lab/cn_grpo/models/Qwen2.5-7B-Instruct"

In [10]:
engineA_constraints_path = os.path.join(output_path, f"{base_name}_engineA_constraints.txt")
engineA_extraction_path  = os.path.join(output_path, f"{base_name}_engineA_extraction.json")

engineB_constraints_path = os.path.join(output_path, f"{base_name}_engineB_constraints.txt")
engineB_extraction_path  = os.path.join(output_path, f"{base_name}_engineB_extraction.json")

In [11]:
print(engineA_constraints_path)

data/temp/option2/X00016469612_engineA_constraints.txt


## Common Utils

In [12]:
def save_text(text: str, path: str):
    with open(path, "w", encoding="utf-8") as f:
        f.write(text)

In [13]:
def save_json(obj, path: str):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)

In [14]:
def extract_json_block(text: str) -> str:
    match = re.search(r"\{.*\}", text, re.DOTALL)
    if match:
        return match.group(0)
    return text

In [15]:
def try_parse_json(text: str):
    try:
        return json.loads(text)
    except Exception:
        pass

    try:
        block = extract_json_block(text)
        return json.loads(block)
    except Exception:
        return None

In [16]:
def load_model_and_tokenizer(model_path: str):
    tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(
        model_path,
        torch_dtype=torch.float16,
        device_map="auto",
        trust_remote_code=True
    )
    return tokenizer, model

In [17]:
def run_chat_inference(model, tokenizer, prompt: str, max_new_tokens: int = 1200) -> str:
    messages = [
        {"role": "user", "content": prompt}
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False
        )

    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )
    return response

In [18]:
def release_model(model=None, tokenizer=None):
    del model
    del tokenizer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

## Phase A: Inferred Constraints Generation

"""
Step 1: inferred_constraints generation

Goal:
Let the model generate field-specific identification / validation heuristics
for company, date, address, and total based only on OCR raw text.

These inferred_constraints are intended to replace the manually designed
rules in the previous pipeline.
"""

In [68]:
# read raw text
with open(raw_text_path, "r", encoding="utf-8") as f:
    raw_text = f.read()

In [69]:
print(raw_text)

tan woon yann

BOOK TA -K (TAMAN DAYA) SDN BHD
TBO7-W
NO.5? $5,57 & 59, JALAN SAGU 18,
TAMAN DAYA
81100 JOHOR BAHRU.
JOHOR.

LAM MCAT

Document Ho : TDO1167104

Date 25/12/2018 8:13:39 PM
Cashier MANIS
Member
CASH BILL
CODE/DESC PRICE Dise AMOUNT
Quy RM RM
9556939040118 AF MODELLING CLAY KIDDY FiSHt
1PC + 9.00) 0,00 9.00
Total : 9,00
Rour ding Adjustment 0.00
Round. :d Total (RM): 9.00
Cash a 40.00.
CHANGE 00

GOODS SOLD ARE NOT RETURNAP
EXCHANGEABLE

THANK YOU
PLEASE COME AGAIN t



In [131]:
def build_constraints_prompt_A(raw_text):
    return f"""
You are a receipt analysis engine.

You are given OCR raw text extracted from a receipt.

Your goal is to infer GENERAL validation constraints for verifying extracted fields.

Fields:
- company
- date
- address
- total

Instructions:

Step 1.
Briefly analyze what kinds of textual patterns appear in the OCR text
(e.g., numeric patterns, address structure, date patterns).

Step 2.
Based on those observations, infer GENERAL validation constraints
that could apply to many receipts.

Important restrictions:

- Do NOT include layout assumptions (top/bottom/middle).
- Do NOT copy words from this receipt.
- Do NOT invent location-specific rules.
- Constraints must describe textual properties only.

Return ONLY JSON:

{{
  "inferred_constraints": {{
    "company": [],
    "date": [],
    "address": [],
    "total": []
  }}
}}

OCR raw text:
{raw_text}
"""

In [132]:
def build_constraints_prompt_B(raw_text):
    return f"""
You are a receipt analysis engine.

You are given OCR raw text extracted from a receipt.

Your goal is to infer GENERAL validation constraints for verifying extracted fields.

Fields:
- company
- date
- address
- total

Instructions:

Step 1.
Briefly analyze what kinds of textual patterns appear in the OCR text
(e.g., numeric patterns, address structure, date patterns).

Step 2.
Based on those observations, infer GENERAL validation constraints
that could apply to many receipts.

Important restrictions:

- Do NOT include layout assumptions (top/bottom/middle).
- Do NOT copy words from this receipt.
- Do NOT invent location-specific rules.
- Constraints must describe textual properties only.

Use exactly this format:

Step 1: Textual patterns analysis
<Your analysis here>

Step 2: Inferred validation constraints
```json
{{
  "inferred_constraints": {{
    "company": [],
    "date": [],
    "address": [],
    "total": []
  }}
}}

OCR raw text:
{raw_text}
"""

In [82]:
# def build_constraints_prompt(raw_text):
#     return f"""
# You are a receipt analysis engine.

# You are given OCR raw text extracted from a receipt.

# Your goal is to produce a constraint trace for validating these four fields:
# - company
# - date
# - address
# - total

# Your output must contain TWO parts:
# 1. textual_patterns_analysis
# 2. inferred_constraints

# Requirements for textual_patterns_analysis:
# - Summarize useful textual patterns relevant to validating the four target fields.
# - Focus on:
#   - numeric patterns
#   - date/time patterns
#   - address structure
#   - company name characteristics
#   - total amount patterns

# Requirements for inferred_constraints:
# - Based on the textual patterns analysis, infer GENERAL validation constraints
#   that could apply to many receipts.

# Important restrictions:
# - Do NOT include layout assumptions (top/bottom/middle).
# - Do NOT copy specific words from this receipt as constraints.
# - Do NOT invent location-specific rules.
# - Constraints must describe textual properties only.

# Return ONLY valid JSON in exactly this format:

# {{
#   "textual_patterns_analysis": {{
#     "numeric_patterns": [],
#     "date_time_patterns": [],
#     "address_structure": [],
#     "company_name_characteristics": [],
#     "total_amount_patterns": []
#   }},
#   "inferred_constraints": {{
#     "company": [],
#     "date": [],
#     "address": [],
#     "total": []
#   }}
# }}

# OCR raw text:
# {raw_text}
# """

In [114]:
# def build_constraints_prompt_qwen(raw_text):
#     return f"""
# You are a receipt analysis engine.

# Your task is to produce a constraint trace for validating these four fields:
# - company
# - date
# - address
# - total

# Your output must contain exactly TWO sections and nothing else.

# Section A: Textual patterns analysis
# - Summarize only the useful textual patterns relevant to validating the four target fields.
# - Focus on:
#   - numeric patterns
#   - date/time patterns
#   - address structure
#   - company name characteristics
#   - total amount patterns

# Section B: Inferred validation constraints
# - Based on Section A, infer general validation constraints that could apply to many receipts.
# - Constraints must describe textual properties only.

# Important restrictions:
# - Do NOT output "Thinking Process".
# - Do NOT analyze the task instructions.
# - Do NOT explain your plan.
# - Do NOT enumerate OCR lines one by one.
# - Do NOT include layout assumptions such as top/bottom/middle.
# - Do NOT copy specific words from this receipt as constraints.
# - Do NOT invent location-specific rules.

# Use exactly this format:

# Section A: Textual patterns analysis
# <analysis>

# Section B: Inferred validation constraints
# ```json
# {{
#   "inferred_constraints": {{
#     "company": [],
#     "date": [],
#     "address": [],
#     "total": []
#   }}
# }}

# OCR raw text:
# {raw_text}
# """

# def build_constraints_prompt_qwen(raw_text):
#     return f"""
# You are a receipt analysis engine.

# You are given OCR raw text extracted from a receipt.

# Your goal is to produce a constraint trace for validating these four fields:
# - company
# - date
# - address
# - total

# Your output must contain TWO parts:
# 1. textual_patterns_analysis
# 2. inferred_constraints

# Requirements for textual_patterns_analysis:
# - Summarize useful textual patterns relevant to validating the four target fields.
# - Focus on:
#   - numeric patterns
#   - date/time patterns
#   - address structure
#   - company name characteristics
#   - total amount patterns

# Requirements for inferred_constraints:
# - Based on the textual patterns analysis, infer GENERAL validation constraints
#   that could apply to many receipts.

# Important restrictions:
# - Do NOT include layout assumptions (top/bottom/middle).
# - Do NOT copy specific words from this receipt as constraints.
# - Do NOT invent location-specific rules.
# - Constraints must describe textual properties only.

# Return ONLY valid JSON in exactly this format:

# {{
#   "textual_patterns_analysis": {{
#     "numeric_patterns": [],
#     "date_time_patterns": [],
#     "address_structure": [],
#     "company_name_characteristics": [],
#     "total_amount_patterns": []
#   }},
#   "inferred_constraints": {{
#     "company": [],
#     "date": [],
#     "address": [],
#     "total": []
#   }}
# }}

# OCR raw text:
# {raw_text}
# """

In [84]:
# tokenizer, model = load_model_and_tokenizer(engineA)

In [74]:
# constraints_prompt = build_constraints_prompt(raw_text)

In [75]:
# constraints_response = run_chat_inference(
#     model,
#     tokenizer,
#     constraints_prompt,
#     max_new_tokens=800
# )

# print(constraints_response)

In [76]:
# constraints_json_text = extract_json_block(constraints_response)
# print(constraints_json_text)

# engineA_constraints = json.loads(constraints_json_text)

In [77]:
# print(engineA_constraints)

In [78]:
# save_json(engineA_constraints, engineA_constraints_path)

In [133]:
def generate_constraints(build_constraints_prompt, model_path, raw_text):

    # build prompt
    prompt = build_constraints_prompt(raw_text)

    # load model
    tokenizer, model = load_model_and_tokenizer(model_path)

    # inference
    response = run_chat_inference(
        model,
        tokenizer,
        prompt,
        max_new_tokens=800
    )

    # free GPU
    del model
    del tokenizer
    torch.cuda.empty_cache()
    # release_model(model, tokenizer)

    return response

#### Engine A

In [134]:
engineA_constraints = generate_constraints(build_constraints_prompt_A, engineA, raw_text)

Loading weights: 100%|██████████| 291/291 [00:04<00:00, 69.55it/s, Materializing param=model.norm.weight]                              
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


In [135]:
print(engineA_constraints)

Based on the OCR raw text, here are the observations and inferred validation constraints:

**Step 1: Textual patterns analysis**

- **Numeric patterns:**
  - Date: The date pattern appears to be in the format "DD/MM/YYYY" (e.g., "25/12/2018").
  - Total: The total amount is represented as a decimal number with a comma as the thousand separator (e.g., "9,00").
  - Address: No specific numeric pattern is observed in the address.
  - Company: No specific numeric pattern is observed in the company name.
- **Address structure:**
  - Address lines are separated by a newline character.
  - Address components (street, city, state, zip) are separated by commas.
- **Date patterns:**
  - The date is always in the format "DD/MM/YYYY".
- **Company patterns:**
  - Company names are typically in title case (first letter of each word capitalized).
- **Total patterns:**
  - Total amount is represented as a decimal number with a comma as the thousand separator.

**Step 2: Inferred validation constraints

In [139]:
save_text(engineA_constraints, engineA_constraints_path)

#### Engine B

In [136]:
engineB_constraints = generate_constraints(build_constraints_prompt_B, engineB, raw_text)

Loading weights: 100%|██████████| 339/339 [00:04<00:00, 77.60it/s, Materializing param=model.norm.weight]                              


In [137]:
print(engineB_constraints)

Step 1: Textual patterns analysis
- **Company**: The company name appears at the top and is formatted as a full name with parentheses and suffixes (e.g., "(TAMAN DAYA) SDN BHD").
- **Date**: The date is in the format `DD/MM/YYYY HH:MM:SS PM` (e.g., `25/12/2018 8:13:39 PM`).
- **Address**: The address consists of multiple lines, including house numbers, street names, and city information (e.g., `NO.57 55, JALAN SAGU 18, TAMAN DAYA`).
- **Total**: The total amount is always prefixed with the word "Total" and ends with the currency symbol (e.g., `Total : 9.00`).

Step 2: Inferred validation constraints
```json
{
  "inferred_constraints": {
    "company": [
      "contains_alphanumeric",
      "ends_with_sdn_bhd_or_similar"
    ],
    "date": [
      "matches_pattern-DD-MM-YYYY_HH:MM:SS_PM"
    ],
    "address": [
      "multiple_lines",
      "includes_house_number_and_street_name",
      "includes_city_information"
    ],
    "total": [
      "starts_with_total",
      "ends_with_currenc

In [138]:
save_text(engineB_constraints, engineB_constraints_path)

## Phase B: Constraint-guided Extraction

In [142]:
with open(raw_text_path, "r", encoding="utf-8") as f:
    raw_text = f.read()

with open(engineA_constraints_path, "r", encoding="utf-8") as f:
    engineA_constraint_trace = f.read()

with open(engineB_constraints_path, "r", encoding="utf-8") as f:
    engineB_constraint_trace = f.read()

In [150]:
def build_extraction_prompt(raw_text, constraint_trace):
    return f"""
You are a receipt extraction engine.

You are given:
1. OCR raw text from a receipt
2. A constraint trace previously generated for this receipt

Your task is to extract these four fields:
- company
- date
- address
- total

You must use the constraint trace to guide your extraction.

For each field, output three things:
1. field_extraction: the final extracted value
2. evidence_trace: the most relevant supporting OCR text
3. reasoning: explain why this evidence supports the extraction, how the constraint trace helped, why competing candidates were not selected if relevant, and any remaining uncertainty

Return ONLY valid JSON in exactly this format:

{{
  "field_extraction": {{
    "company": "",
    "date": "",
    "address": "",
    "total": ""
  }},
  "evidence_trace": {{
    "company": {{
      "evidence_text": ""
    }},
    "date": {{
      "evidence_text": ""
    }},
    "address": {{
      "evidence_text": ""
    }},
    "total": {{
      "evidence_text": ""
    }}
  }},
  "reasoning": {{
    "company": "",
    "date": "",
    "address": "",
    "total": ""
  }}
}}

Rules:
- Use only the OCR raw text and the constraint trace below.
- Do not use outside knowledge.
- If a field is uncertain, still provide the best guess, but mention the uncertainty in reasoning.
- Keep evidence_trace short and grounded in the OCR text.
- If the selected field value or supporting OCR evidence contains "?", preserve it exactly as it appears in the OCR text.
- Do NOT remove, replace, or normalize question marks.
- Do NOT guess missing characters hidden by "?".
- Do not wrap the output in markdown or code fences.

Constraint trace:
{constraint_trace}

OCR raw text:
{raw_text}
"""

In [151]:
def run_extraction_with_constraints(model_path, raw_text, constraint_trace, max_new_tokens=1600):
    prompt = build_extraction_prompt(raw_text, constraint_trace)

    tokenizer, model = load_model_and_tokenizer(model_path)

    response = run_chat_inference(
        model,
        tokenizer,
        prompt,
        max_new_tokens=max_new_tokens
    )

    extraction_json_text = extract_json_block(response)
    extraction_json = json.loads(extraction_json_text)

    del model
    del tokenizer
    torch.cuda.empty_cache()

    return extraction_json

#### EngineA

In [152]:
engineA_extraction = run_extraction_with_constraints(
    model_path=engineA,
    raw_text=raw_text,
    constraint_trace=engineA_constraint_trace,
    max_new_tokens=1600
)

Loading weights: 100%|██████████| 291/291 [00:03<00:00, 90.92it/s, Materializing param=model.norm.weight]                               
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


In [153]:
print(engineA_extraction)

{'field_extraction': {'company': 'BOOK TA -K (TAMAN DAYA) SDN BHD', 'date': '25/12/2018', 'address': 'NO.5? $5,57 & 59, JALAN SAGU 18, TAMAN DAYA 81100 JOHOR BAHRU.', 'total': '9.00'}, 'evidence_trace': {'company': {'evidence_text': 'BOOK TA -K (TAMAN DAYA) SDN BHD'}, 'date': {'evidence_text': 'Date 25/12/2018'}, 'address': {'evidence_text': 'NO.5? $5,57 & 59, JALAN SAGU 18, TAMAN DAYA 81100 JOHOR BAHRU.'}, 'total': {'evidence_text': 'Total : 9,00'}}, 'reasoning': {'company': 'The company name is in title case and matches the pattern of having a maximum length of 50 characters. The evidence text is the company name itself.', 'date': "The date pattern matches the observed format of 'DD/MM/YYYY'. The evidence text is the date itself.", 'address': "The address is separated by a newline character and has a minimum length of 10 characters. The evidence text is the address itself. However, there is some uncertainty due to the presence of '?' in the address.", 'total': 'The total amount is re

In [154]:
save_json(engineA_extraction, engineA_extraction_path)

#### EngineB

In [155]:
engineB_extraction = run_extraction_with_constraints(
    model_path=engineB,
    raw_text=raw_text,
    constraint_trace=engineB_constraint_trace,
    max_new_tokens=1600
)

Loading weights: 100%|██████████| 339/339 [00:03<00:00, 109.17it/s, Materializing param=model.norm.weight]                              


In [156]:
print(engineB_extraction)

{'field_extraction': {'company': 'TAMAN DAYA) SDN BHD', 'date': '25/12/2018 8:13:39 PM', 'address': 'NO.57 55, JALAN SAGU 18, TAMAN DAYA\n81100 JOHOR BAHRU.', 'total': 'Total : 9.00'}, 'evidence_trace': {'company': {'evidence_text': '(TAMAN DAYA) SDN BHD'}, 'date': {'evidence_text': 'Date 25/12/2018 8:13:39 PM'}, 'address': {'evidence_text': 'NO.57 55, JALAN SAGU 18,\nTAMAN DAYA\n81100 JOHOR BAHRU.'}, 'total': {'evidence_text': 'Total : 9,00'}}, 'reasoning': {'company': "The company name is extracted from the top of the receipt, matching the pattern of containing alphanumeric characters and ending with 'SDN BHD'. The evidence text directly supports this extraction.", 'date': 'The date is clearly marked on the receipt and matches the specified pattern. The evidence text provides the exact date and time.', 'address': 'The address is found in the middle section of the receipt, consisting of multiple lines with house numbers, street names, and city information. The evidence text aligns wit

In [157]:
save_json(engineB_extraction, engineB_extraction_path)

# 3-Evaluate

In [32]:
eval_model = "/projects/insightx-lab/cn_grpo/models/Llama-3.1-8B-Instruct"

In [33]:
eval_model2 = "/projects/insightx-lab/cn_grpo/models/Qwen3.5-9B"

In [34]:
eval_model3 = "/projects/insightx-lab/cn_grpo/models/Mistral-7B-Instruct-v0.3"

### Stage1-Rule Synthesis

In [35]:
eval_step1_path = os.path.join(
    output_path,
    f"{base_name}_eval_constraints.json"
)

In [36]:
with open(engineA_constraints_path, "r", encoding="utf-8") as f:
    engineA_constraints_text = f.read()

with open(engineB_constraints_path, "r", encoding="utf-8") as f:
    engineB_constraints_text = f.read()

In [22]:
# def extract_json_block(text):
#     match = re.search(r"```json\s*(.*?)\s*```", text, re.DOTALL)
#     if match:
#         return match.group(1)
#     return None

In [23]:
# jsonA = extract_json_block(engineA_constraints_text)
# print(jsonA)

In [24]:
# jsonB = extract_json_block(engineB_constraints_text)
# print(jsonB)

In [56]:
# def build_rule_synthesis_prompt(engineA_constraints_text, engineB_constraints_text):
#     return f"""

# You are a rule synthesis engine for receipt field validation.

# Two extraction engines have independently generated validation constraints for these fields:
# - company
# - date
# - address
# - total

# Your task is to synthesize the two sets of constraints into ONE consolidated set of checking constraints.

# Goal:
# - absorb useful constraints from both engines
# - resolve disagreements
# - prefer generalizable constraints over overly strict or overly literal ones
# - avoid receipt-specific wording
# - produce constraints that are useful for checking field extraction quality later

# Important instructions:
# - Do NOT simply concatenate the two constraint sets.
# - Do NOT copy receipt-specific phrases as final constraints.
# - Do NOT preserve overly strict regex-like constraints if they are too brittle for OCR text.
# - When two engines disagree, prefer a more robust and general checking rule.
# - The final constraints should help evaluate whether extracted fields are plausible and grounded.

# Return ONLY valid JSON in exactly this format:

# {{
#   "consolidated_constraints": {{
#     "company": [],
#     "date": [],
#     "address": [],
#     "total": []
#   }},
#   "synthesis_summary": {{
#     "agreement_level": "",
#     "notes": ""
#   }}
# }}

# Engine A constraints:
# {engineA_constraints_text}

# Engine B constraints:
# {engineB_constraints_text}
# """

In [59]:
# def build_rule_synthesis_prompt(engineA_constraints_text, engineB_constraints_text):
#     return f"""
# You are a rule synthesis engine.

# You may briefly analyze the inputs, but limit reasoning to at most 5 short bullet points.
# After that, immediately output the final JSON.

# Do not produce long chain-of-thought.

# Return ONLY valid JSON in this format:

# {{
#   "consolidated_constraints": {{
#     "company": [],
#     "date": [],
#     "address": [],
#     "total": []
#   }},
#   "synthesis_summary": {{
#     "agreement_level": "",
#     "notes": ""
#   }}
# }}

# Engine A constraints:
# {engineA_constraints_text}

# Engine B constraints:
# {engineB_constraints_text}
# """

In [25]:
# def build_rule_synthesis_prompt(engineA_constraints_text, engineB_constraints_text):
#     return f"""
# You are a rule synthesis engine for receipt field validation.

# Your task is to synthesize ONE shared set of consolidated checking rules for these four fields:
# - company
# - date
# - address
# - total

# You are given two engine outputs.
# Each engine output may contain:
# 1. textual pattern analysis
# 2. inferred validation constraints
# You should use BOTH parts as input evidence.

# Goal:
# Produce consolidated checking rules that can later be used as a shared evaluation standard for cross-engine judgment.

# Important instructions:
# - Use both engines' analysis and inferred constraints.
# - Do not simply concatenate the two inputs.
# - Resolve disagreements by preferring rules that are more robust, more generalizable, and more useful for later evaluation.
# - Keep the rules useful for checking plausibility and grounding of extracted fields.
# - Avoid overly brittle or overly narrow rules that may fail under OCR noise.
# - Avoid receipt-specific wording unless it is genuinely useful as a general checking clue.
# - The output rules should be field-level checking rules, not raw extraction outputs.
# - Preserve useful structural clues such as format, typical content type, positional hints, and semantic characteristics when supported by the inputs.

# Return ONLY valid JSON in the following format:

# {{
#   "consolidated_rules": {{
#     "company": [],
#     "date": [],
#     "address": [],
#     "total": []
#   }},
#   "synthesis_summary": {{
#     "agreement_level": "high | moderate | low",
#     "notes": "short summary of how the two engine outputs were consolidated"
#   }}
# }}

# Guidelines for consolidated_rules:
# - Each field should contain a short list of compact checking rules.
# - A checking rule can be:
#   - a short natural-language rule string, or
#   - a compact structured object if necessary.
# - Prefer short natural-language rules unless structure is clearly beneficial.

# Engine A output:
# {engineA_constraints_text}

# Engine B output:
# {engineB_constraints_text}
# """

In [28]:
def build_rule_synthesis_prompt(engineA_constraints_text: str, engineB_constraints_text: str) -> str:
    return f"""
Return ONLY valid JSON.
Do not output explanations.

You are a rule synthesis engine for receipt field evaluation.

Two extraction engines independently analyzed the OCR text and produced:
- textual pattern analysis
- inferred validation constraints

Your task is to synthesize these into ONE shared set of consolidated checking rules
for evaluating the credibility of extracted fields.

Fields:
- company
- date
- address
- total

These rules will later be used for self-justification and cross-engine judgment.

--------------------------------
SYNTHESIS PRINCIPLES
--------------------------------

1. Do NOT simply merge or concatenate rules from both engines.

2. If two rules express similar ideas, merge them into ONE generalized rule.

3. Prefer generalized plausibility rules rather than strict schema validation.

4. Avoid overly brittle rules that depend on a specific OCR instance.

5. Preserve useful signals such as:
   - format clues
   - semantic meaning
   - typical receipt layout hints
   - numeric or textual patterns

6. The goal is **evaluation-oriented checking rules**, not strict regex validation.

7. Each field should contain **3–5 compact checking rules**.

--------------------------------
GOOD RULE EXAMPLES
--------------------------------

Good:
"should resemble a plausible merchant name"
"commonly follows a DD/MM/YYYY-style format"
"may include decimal digits and optional currency notation"

Bad:
"pattern must be ^\\d{{2}}/\\d{{2}}/\\d{{4}}$"
"max length must be 50"

--------------------------------
OUTPUT FORMAT
--------------------------------

Return JSON in EXACTLY this format:

{{
  "consolidated_rules": {{
    "company": [],
    "date": [],
    "address": [],
    "total": []
  }},
  "synthesis_summary": {{
    "agreement_level": "high | moderate | low",
    "notes": ""
  }}
}}

--------------------------------
ENGINE A OUTPUT
--------------------------------

{engineA_constraints_text}

--------------------------------
ENGINE B OUTPUT
--------------------------------

{engineB_constraints_text}
"""

In [27]:
# def synthesize_constraints(eval_model_path, engineA_constraints_text, engineB_constraints_text, max_new_tokens=2400):
#     prompt = build_rule_synthesis_prompt(
#         engineA_constraints_text,
#         engineB_constraints_text
#     )

#     tokenizer, model = load_model_and_tokenizer(eval_model_path)

#     response = run_chat_inference(
#         model,
#         tokenizer,
#         prompt,
#         max_new_tokens=max_new_tokens
#     )

#     result_json_text = extract_json_block(response)

#     print("===== RAW RESPONSE =====")
#     print(response)
#     print("===== EXTRACTED JSON TEXT =====")
#     print(result_json_text)
    
#     result = json.loads(result_json_text)

#     del model
#     del tokenizer
#     torch.cuda.empty_cache()

#     return result, response

In [29]:
# prompt = build_rule_synthesis_prompt(
#     engineA_constraints_text,
#     engineB_constraints_text
# )

In [30]:
# tokenizer, model = load_model_and_tokenizer(eval_model)

# response = run_chat_inference(
#     model,
#     tokenizer,
#     prompt,
#     max_new_tokens=3600
# )

# print(response)

Loading weights: 100%|██████████| 291/291 [00:09<00:00, 30.13it/s, Materializing param=model.norm.weight]                              
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


{
  "consolidated_rules": {
    "company": [
      "should resemble a plausible merchant name",
      "formatted as a full name with optional parentheses and suffixes",
      "contains alphanumeric characters"
    ],
    "date": [
      "commonly follows a DD/MM/YYYY-style format",
      "may include time in the format HH:MM:SS PM"
    ],
    "address": [
      "may consist of multiple lines",
      "includes house number and street name",
      "includes city information"
    ],
    "total": [
      "may include decimal digits and optional currency notation",
      "starts with the word 'Total'",
      "ends with a currency symbol"
    ]
  },
  "synthesis_summary": {
    "agreement_level": "high",
    "notes": ""
  }
}


In [39]:
def synthesize_constraints(eval_model_path, engineA_constraints_text, engineB_constraints_text, max_new_tokens=3600):
    prompt = build_rule_synthesis_prompt(
        engineA_constraints_text,
        engineB_constraints_text
    )

    tokenizer, model = load_model_and_tokenizer(eval_model)

    raw_response = run_chat_inference(
        model,
        tokenizer,
        prompt,
        max_new_tokens=max_new_tokens
    )

    parsed_json = extract_json_block(raw_response)
    result = json.loads(parsed_json)

    del model
    del tokenizer
    torch.cuda.empty_cache()

    return result

In [40]:
synthesis_constraints = synthesize_constraints(eval_model, engineA_constraints_text, engineB_constraints_text)
print(synthesis_constraints)

Loading weights: 100%|██████████| 291/291 [00:10<00:00, 28.09it/s, Materializing param=model.norm.weight]                              
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


{'consolidated_rules': {'company': ['should resemble a plausible merchant name', 'formatted as a full name with optional parentheses and suffixes', 'contains alphanumeric characters'], 'date': ['commonly follows a DD/MM/YYYY-style format', 'may include time in the format HH:MM:SS PM'], 'address': ['may consist of multiple lines', 'includes house number and street name', 'includes city information'], 'total': ['may include decimal digits and optional currency notation', "starts with the word 'Total'", 'ends with a currency symbol']}, 'synthesis_summary': {'agreement_level': 'high', 'notes': ''}}


In [41]:
save_json(synthesis_constraints, eval_step1_path)

### Stage2-Engine Self-Justification

In [37]:
engineA_stage2_path = os.path.join(
    output_path,
    f"{base_name}_engineA_self_judge.json"
)

In [38]:
engineB_stage2_path = os.path.join(
    output_path,
    f"{base_name}_engineB_self_judge.json"
)

In [39]:
with open(eval_step1_path, "r", encoding="utf-8") as f:
    consolidated_rules = json.load(f)

In [40]:
with open(engineA_extraction_path, "r", encoding="utf-8") as f:
    engineA_extraction = json.load(f)

with open(engineB_extraction_path, "r", encoding="utf-8") as f:
    engineB_extraction = json.load(f)

In [41]:
with open(raw_text_path, "r", encoding="utf-8") as f:
    ocr_raw_text = f.read()

In [26]:
def run_sanity_check(value):

    if value is None:
        return "not_pass"

    value = str(value).strip()

    if value == "":
        return "not_pass"

    if "?" in value:
        return "not_pass"

    return "pass"

In [27]:
engineA_sanity = {}

for field, value in engineA_extraction["field_extraction"].items():
    
    result = run_sanity_check(value)

    engineA_sanity[field] = {
        "value": value,
        "sanity_check_result": result
    }
    
engineA_sanity

{'company': {'value': 'BOOK TA -K (TAMAN DAYA) SDN BHD',
  'sanity_check_result': 'pass'},
 'date': {'value': '25/12/2018', 'sanity_check_result': 'pass'},
 'address': {'value': 'NO.5? $5,57 & 59, JALAN SAGU 18, TAMAN DAYA 81100 JOHOR BAHRU.',
  'sanity_check_result': 'not_pass'},
 'total': {'value': '9.00', 'sanity_check_result': 'pass'}}

In [28]:
engineB_sanity = {}

for field, value in engineB_extraction["field_extraction"].items():
    
    result = run_sanity_check(value)

    engineB_sanity[field] = {
        "value": value,
        "sanity_check_result": result
    }

engineB_sanity

{'company': {'value': 'TAMAN DAYA) SDN BHD', 'sanity_check_result': 'pass'},
 'date': {'value': '25/12/2018 8:13:39 PM', 'sanity_check_result': 'pass'},
 'address': {'value': 'NO.57 55, JALAN SAGU 18, TAMAN DAYA\n81100 JOHOR BAHRU.',
  'sanity_check_result': 'pass'},
 'total': {'value': 'Total : 9.00', 'sanity_check_result': 'pass'}}

In [29]:
# def build_stage2_prompt(
#     field_name,
#     rules,
#     extracted_value,
#     evidence_text,
#     reasoning_text,
#     ocr_raw_text,
#     sanity_result
# ):
    
#     prompt = f"""
# You are an evaluation model for receipt field extraction reliability.

# Your job is to evaluate whether the extracted value for one field is credible.

# Return ONLY valid JSON. No explanation.

# Field name:
# {field_name}

# Consolidated validation rules:
# {json.dumps(rules, indent=2)}

# Extracted value:
# {extracted_value}

# Evidence trace:
# {evidence_text}

# Engine reasoning:
# {reasoning_text}

# OCR raw text:
# {ocr_raw_text}

# Sanity check result:
# {sanity_result}

# Return JSON in exactly this format:

# {{
#   "extracted_value": "",
#   "rule_consistency": "high | moderate | low",
#   "engine_self_consistency": "strong | moderate | weak",
#   "ocr_alignment": "strong | partial | weak",
#   "sanity_check_result": "pass | not_pass",
#   "judgment_summary": ""
# }}
# """

#     return prompt

In [30]:
def build_stage2_prompt(
    field_name,
    rules,
    extracted_value,
    evidence_text,
    reasoning_text,
    ocr_raw_text,
    sanity_result
):
    
    prompt = f"""
You are an evaluation model for receipt field extraction reliability.

Your job is to evaluate whether the extracted value for one field is credible.

Return ONLY valid JSON. No explanation.

Field name:
{field_name}

Consolidated validation rules:
{json.dumps(rules, indent=2, ensure_ascii=False)}

Extracted value:
{extracted_value}

Evidence trace:
{evidence_text}

Engine reasoning:
{reasoning_text}

OCR raw text:
{ocr_raw_text}

Sanity check result:
{sanity_result}

Evaluation definitions:

1. rule_consistency:
How well the extracted value matches the consolidated validation rules for this field.

2. engine_self_consistency:
How well the engine's evidence trace and reasoning support its own extracted value.

3. ocr_alignment:
How well the extracted value is supported by OCR text in a context relevant to the target field.
Do NOT judge this only by whether the value appears somewhere in OCR text.
A value may appear in OCR text but still be weakly aligned if it belongs to a different field context.

Examples:
- For the total field, a numeric value should be supported by total-related context, not just by appearing anywhere among other numbers.
- For the date field, a date-like value should be supported by date-related context, not just any numeric string.
- For the company field, the value should align with merchant-like text.
- For the address field, the value should align with address-like multi-line location text.

4. sanity_check_result:
Copy the provided sanity_check_result exactly.

5. judgment_summary:
Write one short summary sentence explaining the overall credibility of the extracted value.
The summary must be consistent with the other output fields.
Do not introduce new evidence or new conclusions beyond:
- rule_consistency
- engine_self_consistency
- ocr_alignment
- sanity_check_result

Return JSON in exactly this format:

{{
  "extracted_value": "",
  "rule_consistency": "high | moderate | low",
  "engine_self_consistency": "strong | moderate | weak",
  "ocr_alignment": "strong | partial | weak",
  "sanity_check_result": "pass | not_pass",
  "judgment_summary": ""
}}
"""
    return prompt

In [31]:
def evaluate_field(eval_model, field_name, engine_extraction):

    extracted_value = engine_extraction["field_extraction"][field_name]

    evidence_text = engine_extraction["evidence_trace"][field_name]["evidence_text"]

    reasoning_text = engine_extraction["reasoning"][field_name]

    rules = consolidated_rules["consolidated_rules"][field_name]

    sanity_result = run_sanity_check(extracted_value)

    prompt = build_stage2_prompt(
        field_name,
        rules,
        extracted_value,
        evidence_text,
        reasoning_text,
        ocr_raw_text,
        sanity_result
    )

    tokenizer, model = load_model_and_tokenizer(eval_model)

    raw_response = run_chat_inference(
        model,
        tokenizer,
        prompt,
        max_new_tokens=600
    )

    parsed_json = extract_json_block(raw_response)
    result = json.loads(parsed_json)

    del model
    del tokenizer
    torch.cuda.empty_cache()

    return result

In [60]:
# result = evaluate_field(
#     eval_model,
#     "company",
#     engineB_extraction
# )

# print(json.dumps(result, indent=2))

In [32]:
fields = ["company", "date", "address", "total"]

engineA_stage2 = {
    "engine": "engineA",
    "field_results": {}
}

for field in fields:
    
    print("Running", field)

    result = evaluate_field(
        eval_model,
        field,
        engineA_extraction
    )

    engineA_stage2["field_results"][field] = result

Running company


`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 291/291 [00:03<00:00, 93.89it/s, Materializing param=model.norm.weight]                               
The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Running date


Loading weights: 100%|██████████| 291/291 [00:03<00:00, 92.16it/s, Materializing param=model.norm.weight]                               
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Running address


Loading weights: 100%|██████████| 291/291 [00:03<00:00, 92.30it/s, Materializing param=model.norm.weight]                               
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Running total


Loading weights: 100%|██████████| 291/291 [00:03<00:00, 92.29it/s, Materializing param=model.norm.weight]                               
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


In [33]:
save_json(engineA_stage2, engineA_stage2_path)

In [34]:
engineB_stage2 = {
    "engine": "engineB",
    "field_results": {}
}

for field in fields:
    
    print("Running", field)

    result = evaluate_field(
        eval_model,
        field,
        engineB_extraction
    )

    engineB_stage2["field_results"][field] = result

Running company


Loading weights: 100%|██████████| 291/291 [00:03<00:00, 92.23it/s, Materializing param=model.norm.weight]                               
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Running date


Loading weights: 100%|██████████| 291/291 [00:03<00:00, 92.93it/s, Materializing param=model.norm.weight]                               
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Running address


Loading weights: 100%|██████████| 291/291 [00:03<00:00, 92.75it/s, Materializing param=model.norm.weight]                               
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Running total


Loading weights: 100%|██████████| 291/291 [00:02<00:00, 101.10it/s, Materializing param=model.norm.weight]                              
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


In [35]:
save_json(engineB_stage2, engineB_stage2_path)

### Stage3-cross-judgment

In [42]:
stage3_path = os.path.join(
    output_path,
    f"{base_name}_cross_judge.json"
)

In [43]:
with open(engineA_stage2_path, "r", encoding="utf-8") as f:
    engineA_stage2 = json.load(f)

with open(engineB_stage2_path, "r", encoding="utf-8") as f:
    engineB_stage2 = json.load(f)

In [44]:
def cross_judge_prompt(
    field_name,
    consolidated_rules_for_field,
    engineA_field_result,
    engineB_field_result
):
    return f"""
Return ONLY valid JSON.
Do not output explanations.

You are a cross-engine judgment model for receipt field extraction reliability.

Your task is to compare the stage-2 self-justification results from engineA and engineB
for one field, and decide which extracted value is more credible.

Field name:
{field_name}

Consolidated checking rules:
{json.dumps(consolidated_rules_for_field, indent=2, ensure_ascii=False)}

EngineA result:
{json.dumps(engineA_field_result, indent=2, ensure_ascii=False)}

EngineB result:
{json.dumps(engineB_field_result, indent=2, ensure_ascii=False)}

Judgment instructions:
1. Compare the two extracted values and their stage-2 signals.
2. Select the more credible value.
3. If both are unreliable, selected_engine can be "none".
4. final_rule_consistency should reflect the final recommended value.
5. final_engine_self_consistency should reflect how trustworthy the selected engine's self-justification is.
6. final_ocr_alignment should reflect the selected value's OCR support.
7. field_confidence should be one of:
   - very_high
   - high
   - medium
   - low
8. field_state should be one of:
   - pass
   - review_needed
   - fail

Guidance:
- Prefer values with stronger rule consistency, stronger self-consistency, stronger OCR alignment, and passing sanity checks.
- If a value has sanity_check_result = not_pass, treat it as suspicious.
- If both values are weak or unsupported, use selected_engine = "none".
- judgment should be field-aware, not based only on string matching.

Return JSON in exactly this format:

{{
  "recommended_value": "",
  "selected_engine": "engineA | engineB | none",
  "selection_reason": "",
  "final_rule_consistency": "high | moderate | low",
  "final_engine_self_consistency": "strong | moderate | weak",
  "final_ocr_alignment": "strong | partial | weak",
  "field_confidence": "very_high | high | medium | low",
  "field_state": "pass | review_needed | fail"
}}
"""

In [45]:
def evaluate_cross_judgment_field(eval_model, field_name):
    engineA_field_result = engineA_stage2["field_results"][field_name]
    engineB_field_result = engineB_stage2["field_results"][field_name]
    rules = consolidated_rules["consolidated_rules"][field_name]

    prompt = cross_judge_prompt(
        field_name=field_name,
        consolidated_rules_for_field=rules,
        engineA_field_result=engineA_field_result,
        engineB_field_result=engineB_field_result
    )

    tokenizer, model = load_model_and_tokenizer(eval_model)

    raw_response = run_chat_inference(
        model,
        tokenizer,
        prompt,
        max_new_tokens=3600
    )

    parsed_json = extract_json_block(raw_response)
    result = json.loads(parsed_json)

    del model
    del tokenizer
    torch.cuda.empty_cache()

    if result is None:
        raise ValueError(f"Failed to extract valid Stage 3 JSON for field: {field_name}")

    return result

In [46]:
test_stage3 = evaluate_cross_judgment_field(
    eval_model=eval_model,
    field_name="company"
)

print(test_stage3)

`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 291/291 [00:25<00:00, 11.20it/s, Materializing param=model.norm.weight]                              
The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


{'recommended_value': 'BOOK TA -K (TAMAN DAYA) SDN BHD', 'selected_engine': 'engineA', 'selection_reason': "Engine A's extracted value is more consistent with the validation rules and has stronger OCR alignment.", 'final_rule_consistency': 'high', 'final_engine_self_consistency': 'strong', 'final_ocr_alignment': 'strong', 'field_confidence': 'high', 'field_state': 'pass'}


In [47]:
fields = ["company", "date", "address", "total"]

stage3_result = {
    "field_results": {}
}

for field in fields:
    print("Running Stage 3 for", field)
    result = evaluate_cross_judgment_field(
        eval_model=eval_model,
        field_name=field
    )
    stage3_result["field_results"][field] = result

Running Stage 3 for company


Loading weights: 100%|██████████| 291/291 [00:05<00:00, 57.81it/s, Materializing param=model.norm.weight]                              
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Running Stage 3 for date


Loading weights: 100%|██████████| 291/291 [00:05<00:00, 57.87it/s, Materializing param=model.norm.weight]                              
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Running Stage 3 for address


Loading weights: 100%|██████████| 291/291 [00:05<00:00, 57.77it/s, Materializing param=model.norm.weight]                              
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Running Stage 3 for total


Loading weights: 100%|██████████| 291/291 [00:05<00:00, 54.29it/s, Materializing param=model.norm.weight]                              
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


In [48]:
save_json(stage3_result, stage3_path)

### Compress constraints

In [33]:
def build_compress_constraints_prompt(constraints_text):
    return f"""
You are a constraint compression engine for receipt field validation.

You are given one engine's constraints for these fields:
- company
- date
- address
- total

Your task is to compress them into a shorter, structured summary that preserves
the most useful validation information for later cross-engine synthesis.

Important instructions:
- Preserve the useful meaning of the constraints.
- Remove redundant wording.
- Prefer concise and generalizable summaries.
- Do NOT invent new constraints that are not supported by the input.
- Do NOT output thinking process or explanation.
- Output JSON only.

Return ONLY valid JSON in exactly this format:

{{
  "constraint_summary": {{
    "company": [],
    "date": [],
    "address": [],
    "total": []
  }}
}}

Input constraints:
{constraints_text}
"""

In [40]:
def run_chat_inference_qwen(model, tokenizer, prompt: str, max_new_tokens: int = 1600) -> str:
    messages = [
        {
            "role": "system",
            "content": (
                "You are a precise JSON generation engine. "
                "Do not output thinking process, reasoning steps, draft notes, "
                "or any explanation. Directly output the final JSON object only."
            )
        },
        {
            "role": "user",
            "content": prompt
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False
        )

    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )

    return response

In [46]:
prompt = build_compress_constraints_prompt(engineB_constraints_text)

tokenizer, model = load_model_and_tokenizer(eval_model)

response = run_chat_inference_qwen(
    model,
    tokenizer,
    prompt,
    max_new_tokens=2400
)

print(response)

Loading weights: 100%|██████████| 427/427 [00:05<00:00, 83.90it/s, Materializing param=model.norm.weight]                               
Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.


Thinking Process:

1.  **Analyze the Request:**
    *   Role: Constraint compression engine for receipt field validation.
    *   Input: One engine's constraints for fields (company, date, address, total) in two steps (Textual patterns analysis and Inferred validation constraints).
    *   Task: Compress the constraints into a shorter, structured summary preserving useful validation information.
    *   Output Format: Valid JSON only, specifically matching the provided schema: `{"constraint_summary": {"company": [], "date": [], "address": [], "total": []}}`.
    *   Constraints: No thinking process, no explanation, no extra text, no invented constraints.

2.  **Analyze the Input Constraints:**
    *   **Company**:
        *   Textual: Top of receipt, full name, parentheses, suffixes (e.g., "(TAMAN DAYA) SDN BHD").
        *   Inferred: `contains_alphanumeric`, `ends_with_sdn_bhd_or_similar`.
    *   **Date**:
        *   Textual: Format `DD/MM/YYYY HH:MM:SS PM` (e.g., `25/12/2018 8:13:

In [47]:
def extract_json_block_qwen(text: str) -> str:
    text = text.strip()

    # 1) If Qwen outputs a </think> tag, keep only the tail after it
    if "</think>" in text:
        text = text.split("</think>")[-1].strip()

    # 2) Remove markdown fences if present
    if "```json" in text:
        start = text.find("```json") + len("```json")
        end = text.rfind("```")
        if end > start:
            text = text[start:end].strip()
            return text

    if "```" in text:
        start = text.find("```") + len("```")
        end = text.rfind("```")
        if end > start:
            text = text[start:end].strip()
            return text

    # 3) Fallback: keep the outermost JSON-looking object
    start = text.find("{")
    end = text.rfind("}")
    if start != -1 and end != -1 and end > start:
        return text[start:end + 1].strip()

    # 4) If nothing found, return raw text for debugging
    return text

In [48]:
result_json_text = extract_json_block_qwen(response)

In [49]:
result = json.loads(result_json_text)

JSONDecodeError: Extra data: line 1 column 80 (char 79)

In [45]:
print(result)

{'constraint_summary': {'company': [{'type': 'string', 'min': 2, 'max': 50, 'pattern': '^[A-Za-z ]+$'}], 'date': [{'type': 'string', 'pattern': '^(\\d{2}/\\d{2}/\\d{4})$'}], 'address': [{'type': 'string', 'min': 10, 'max': 100}], 'total': [{'type': 'number', 'min': 0, 'max': 9999999.99, 'decimals': 2}]}}
